<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 34px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Core landscape &mdash; where it is filed</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        The geographic story: at which international and regional patent offices the antibiotic-resistance families are filed, in total and over time.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; landscape analyses after <span style="color: #be0f05; font-weight: 600;">Riccardo Priore</span>, Centro PATLIB, AREA Science Park
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 680px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What this notebook does</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; Load the shared corpus<br/>Step&nbsp;2 &nbsp;&middot;&nbsp; Retrieve filings and their authorities<br/>Step&nbsp;3 &nbsp;&middot;&nbsp; Filings by international / regional authority<br/>Step&nbsp;4 &nbsp;&middot;&nbsp; WO (PCT) vs EP over time
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 680px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Part 2 of 4 &mdash; run the four notebooks of this module in order.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            Re-running needs <strong>EPO&nbsp;TIP</strong> (it queries PATSTAT&nbsp;PROD via <code>epo.tipdata</code>).
            Each notebook writes what the next one reads; notebook&nbsp;4 assembles the report.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global.
    </div>
</div>

## Step 1 — Load the shared corpus

We read `dataset.xlsx` from notebook 1 and take its `docdb_family_id` list. Every analysis in this module starts exactly here, so the whole course rests on one corpus.

In [ ]:
import pandas as pd
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS211_PAT_PUBLN
import report_kit

patstat = PatstatClient(env='PROD')
db = patstat.orm()

OUT = "2_core_landscape_analyses_output"
import os; os.makedirs(OUT, exist_ok=True)

dataset = pd.read_excel("1_dataset_and_search_strategy_output/dataset.xlsx")
family_ids = dataset['docdb_family_id'].unique().tolist()
print(f"{len(family_ids):,} families loaded from the shared corpus.")

## Step 2 — Retrieve filings and their authorities

For each family we pull its applications and the **publication authority** (`publn_auth`) — the office where it was filed — together with the application's filing year. We keep one row per distinct *application*, so a family filed at several offices contributes to each. (Note: this counts **filings**, i.e. applications, not families — a family with two EP applications counts twice in the EP bar.)

In [ ]:
rows = (db.query(
        TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id,
        TLS211_PAT_PUBLN.publn_auth, TLS201_APPLN.appln_filing_year)
    .join(TLS211_PAT_PUBLN, TLS201_APPLN.appln_id == TLS211_PAT_PUBLN.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(family_ids))
    .distinct().all())
auth = pd.DataFrame(rows, columns=['docdb_family_id', 'appln_id', 'publn_auth', 'filing_year'])
print(f"{len(auth):,} distinct filings across {auth['publn_auth'].nunique()} authorities.")

## Step 3 — Filings by international / regional authority

We focus on the five **supranational** routes — WO (PCT), EP (European Patent Office), and the regional offices EA (Eurasian), AP (ARIPO), OA (OAPI) — because they show the international ambition of the field. This is the first chart of the notebook.

In [ ]:
import plotly.express as px

intl = ["WO", "EP", "EA", "AP", "OA"]
labels = {"WO": "WO — PCT (WIPO)", "EP": "EP — European Patent Office",
          "EA": "EA — Eurasian Patent Org.", "AP": "AP — ARIPO (Africa)",
          "OA": "OA — OAPI (Africa)"}
intl_df = auth[auth['publn_auth'].isin(intl)]
totals = intl_df['publn_auth'].value_counts().reindex(intl).fillna(0).astype(int)
totals_tbl = pd.DataFrame({'authority': [labels[a] for a in totals.index],
                           'filings': totals.values})

fig = px.bar(totals_tbl, x='authority', y='filings', text='filings')
fig.update_traces(marker_color='#4527A0', textposition='outside')
fig.update_layout(title='Filings by international / regional authority',
                  xaxis_title='Authority', yaxis_title='Number of filings',
                  template='plotly_white', height=500)

report_kit.record(210, "intl_authority_totals",
                  "Filings by international / regional authority", fig, totals_tbl, output_dir=OUT,
                  note="Counts distinct applications (filings), not families. WO/EP/EA/AP/OA.")

## Step 4 — WO (PCT) vs EP over time

The two dominant routes — the global PCT filing (WO) and the European route (EP) — plotted per filing year from 2000 to 2023. It shows how the field's international vs European footprint evolved.

In [ ]:
yearly = (intl_df[intl_df['publn_auth'].isin(['WO', 'EP'])]
          .groupby(['filing_year', 'publn_auth']).size().reset_index(name='filings'))
yearly = yearly[(yearly['filing_year'] >= 2000) & (yearly['filing_year'] <= 2023)]

fig = px.line(yearly, x='filing_year', y='filings', color='publn_auth', markers=True,
              labels={'filing_year': 'Filing year', 'filings': 'Number of filings',
                      'publn_auth': 'Authority'})
fig.update_layout(title='WO (PCT) vs EP filings over time',
                  template='plotly_white', height=500)

report_kit.record(220, "intl_authority_trend",
                  "WO (PCT) vs EP over time", fig, yearly, output_dir=OUT,
                  note="Filings per year for WO and EP, 2000–2023.")

---
**Done.** Two geographic charts recorded. Continue with notebook 3.